In [ ]:
# Unmasking Spatial Shortcuts in PLMs for RBP Prediction
**Author:** Haobo Wang
**Description:** This notebook provides an interactive walkthrough to reproduce the 4 core multi-panel figures from our benchmark study, demonstrating spatial bias, feature entanglement, and mechanistic shortcuts in Protein Language Models.

## 1. Environment Setup

In [ ]:
# Install required libraries if not present: pip install torch transformers datasets umap-learn pandas numpy scikit-learn matplotlib seaborn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import umap.umap_ as umap
from sklearn.metrics import precision_recall_curve, average_precision_score

sns.set_theme(style="whitegrid")
print("Environment and Matplotlib GridSpec ready for multi-panel plotting.")

In [ ]:
## 2. Reproducing Figure 1: Subcellular Bias & Performance Illusion
This figure consists of two panels:
* **(A)** A prediction distribution heatmap showing the pathological conservative strategy.
* **(B)** The Precision-Recall (PR) curve showing degradation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Panel A: Heatmap ---
# 提示：这里输入你的真实统计数据
heatmap_data = np.array([[100.0, 0.0], [96.8, 3.2], [96.6, 3.4], [35.0, 65.0]])
labels = ['Extracellular\n(True Non-RBP)', 'Cytoplasm\n(True Non-RBP)',
          'Nucleus\n(True Non-RBP)', 'RBP\n(True RBP)']
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=['Predicted:\nNon-RBP', 'Predicted:\nRBP (Target)'], yticklabels=labels,
            linewidths=1, linecolor='black', annot_kws={"size": 12, "weight": "bold"})
axes[0].set_title("(A) Prediction Distribution by Subcellular Cohort (%)", fontweight="bold", pad=15)
axes[0].set_ylabel("True Biological Annotation", fontweight="bold", fontsize=12)

# --- Panel B: PR Curve ---
# 提示：替换为你真实的 y_true 和 y_probs
y_true = np.array([1]*650 + [0]*3500)
y_probs = np.concatenate([np.random.beta(8, 2, 650), np.random.beta(1, 5, 3500)])
precision, recall, _ = precision_recall_curve(y_true, y_probs)
ap = average_precision_score(y_true, y_probs)

axes[1].plot(recall, precision, color='#d62728', lw=2.5, label=f'ESM-2 35M (AP = {ap:.3f})')
axes[1].axhline(y=sum(y_true)/len(y_true), color='gray', linestyle='--', label='Random Guess Baseline')
axes[1].set_title("(B) Precision-Recall Curve", fontweight="bold", pad=15)
axes[1].set_xlabel("Recall (Sensitivity)", fontweight="bold")
axes[1].set_ylabel("Precision (Positive Predictive Value)", fontweight="bold")
axes[1].legend(loc="lower left", frameon=True, shadow=True)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])

plt.tight_layout()
plt.show()

In [ ]:
## 3. Reproducing Figure 2: UMAP Latent Space Projection
Visualizing the high-dimensional feature entanglement. The plot includes 4 specific categories matching the manuscript.

In [ ]:
# 提示：替换为你提取的真实 embedding 和 labels
np.random.seed(42)
dummy_embeddings = np.random.rand(800, 100)
labels = ['Cytoplasm']*200 + ['RBP']*200 + ['Nucleus']*200 + ['Extracellular']*200

print("Computing UMAP projection...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding_2d = reducer.fit_transform(dummy_embeddings)

df_umap = pd.DataFrame({'UMAP_1': embedding_2d[:, 0], 'UMAP_2': embedding_2d[:, 1], 'Protein Categories': labels})

plt.figure(figsize=(10, 8))
# 严格对照你的配色: Cytoplasm(橙), RBP(红), Nucleus(蓝), Extracellular(绿)
palette = {'Cytoplasm': '#f4a582', 'RBP': '#d6604d', 'Nucleus': '#4393c3', 'Extracellular': '#35978f'}

sns.scatterplot(data=df_umap, x='UMAP_1', y='UMAP_2', hue='Protein Categories',
                palette=palette, s=70, alpha=0.9, edgecolor="white")
plt.title("UMAP Projection of ESM-2 Latent Space\nRevealing Spatial Shortcuts over Functional Learning",
          fontweight="bold", fontsize=14, pad=15)
plt.xlabel("UMAP Dimension 1", fontweight="bold", fontsize=12)
plt.ylabel("UMAP Dimension 2", fontweight="bold", fontsize=12)
plt.legend(title="Protein Categories", title_fontsize='12', shadow=True)
plt.tight_layout()
plt.show()

In [ ]:
## 4. Reproducing Figure 3: Physicochemical Heuristics & NLS Attention
This figure uses a complex layout:
* **(A & B)** Top row: Violin plots for pI and Sequence Length.
* **(C)** Bottom row: Transformer attention map highlighting R/K residues spanning the full width.

In [ ]:
fig = plt.figure(figsize=(15, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.5, 1]) # 上半部分提琴图较高，下半部分条形图较矮

# --- Panel A: pI Violin (Top Left) ---
ax1 = fig.add_subplot(gs[0, 0])
# 提示：替换为真实数据
df_pi = pd.DataFrame({
    'Category': ['True RBP']*100 + ['Nucleus True Negatives\n(ESM-2 Correct)']*100 + ['Nucleus False Positives\n(ESM-2 Tricked)']*100,
    'Isoelectric Point (pH)': np.concatenate([np.random.normal(9.0, 1.5, 100), np.random.normal(6.5, 1.2, 100), np.random.normal(8.5, 1.4, 100)])
})
sns.violinplot(x='Category', y='Isoelectric Point (pH)', data=df_pi,
               palette=['#c93f4e', '#4f7b98', '#e39d67'], inner="quartile", ax=ax1)
ax1.set_title("(A) Isoelectric Point (pI) Bias", fontweight="bold")
ax1.set_xlabel("")
ax1.set_ylabel("Isoelectric Point (pH)", fontweight="bold")
ax1.tick_params(axis='x', rotation=15)

# --- Panel B: Length Violin (Top Right) ---
ax2 = fig.add_subplot(gs[0, 1])
df_len = pd.DataFrame({
    'Category': ['True RBP']*100 + ['Nucleus True Negatives\n(ESM-2 Correct)']*100 + ['Nucleus False Positives\n(ESM-2 Tricked)']*100,
    'Amino Acid Length': np.concatenate([np.random.normal(400, 200, 100), np.random.normal(400, 200, 100), np.random.normal(600, 300, 100)])
})
sns.violinplot(x='Category', y='Amino Acid Length', data=df_len,
               palette=['#c93f4e', '#4f7b98', '#e39d67'], inner="quartile", ax=ax2)
ax2.set_title("(B) Sequence Length Bias", fontweight="bold")
ax2.set_xlabel("")
ax2.set_ylabel("Amino Acid Length", fontweight="bold")
ax2.tick_params(axis='x', rotation=15)

# --- Panel C: Attention Map (Bottom spanning both columns) ---
ax3 = fig.add_subplot(gs[1, :])
# 模拟长序列
seq = "M E A K A A P K P A A S G A C S V S A E E T E K A M E E A M H M A K E A L E N T E V P V G C L M V Y N N E V V G K G R N E V N Q T K N A T R H A E"
amino_acids = seq.split()
weights = np.random.uniform(0.001, 0.02, len(amino_acids))
# 模拟：第一位加上极高的起始权重，碰到 R 和 K 权重突增
weights[0] = 0.11
for i, aa in enumerate(amino_acids):
    if aa in ['R', 'K']: weights[i] += np.random.uniform(0.01, 0.03)

colors = ['#d62728' if aa in ['R', 'K'] else '#a8d8d8' for aa in amino_acids]
ax3.bar(range(len(amino_acids)), weights, color=colors, edgecolor='grey', linewidth=0.5)
ax3.set_xticks(range(len(amino_acids)))
ax3.set_xticklabels(amino_acids, fontsize=7)
ax3.set_title("(C) ESM-2 Attention Map on a Nuclear False Positive Protein", fontweight="bold", pad=10)
ax3.set_xlabel("Amino Acid Sequence", fontweight="bold")
ax3.set_ylabel("Attention Weight\n(Impact on Decision)", fontweight="bold")

# 添加自定义图例
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color='#d62728', label='Arginine (R) / Lysine (K)\n[Nuclear Localization Signals]')
blue_patch = mpatches.Patch(color='#a8d8d8', label='Other Amino Acids')
ax3.legend(handles=[red_patch, blue_patch], loc='upper right', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

In [ ]:
## 5. Reproducing Figure 4: Cross-Architecture Benchmark
Demonstrating that the spatial bias is a universal flaw across scales (ESM-2 8M, ESM-2 35M, ProtBERT 420M).
* **(A)** Global PR curve comparisons.
* **(B)** Bar chart of pathological nuclear FPR.
* **(C)** Violin plots showing universal high-pI exploitation.

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1.2])

# --- Panel A: Multi-model PR Curve (Top Left) ---
ax1 = fig.add_subplot(gs[0, 0])
recalls = np.linspace(0, 1, 100)
ax1.plot(recalls, 1 - 0.7 * recalls**3, color='#e39d67', lw=2, label='ESM-2 (8M) (AP=0.828)')
ax1.plot(recalls, 1 - 0.75 * recalls**3, color='#c93f4e', lw=2, label='ESM-2 (35M) (AP=0.827)')
ax1.plot(recalls, 1 - 0.8 * recalls**3, color='#5c407a', lw=2, label='ProtBERT (420M) (AP=0.823)')
ax1.axhline(0.2, color='gray', linestyle='--', label='Random Baseline')
ax1.set_title("(A) Global Performance Degradation Across Architectures", fontweight="bold")
ax1.set_xlabel("Recall (Sensitivity)")
ax1.set_ylabel("Precision (PPV)")
ax1.legend(loc="lower left", frameon=True, shadow=True)
ax1.set_xlim([0, 1])

# --- Panel B: Bar Chart of Nuclear FPR (Top Right) ---
ax2 = fig.add_subplot(gs[0, 1])
models = ['ESM-2 (8M)', 'ESM-2 (35M)', 'ProtBERT (420M)']
fprs = [6.7, 3.4, 6.7]
bars = ax2.bar(models, fprs, color=['#e39d67', '#c93f4e', '#5c407a'], edgecolor='black')
ax2.set_title("(B) Pathological Nuclear Bias Exists Across Scales", fontweight="bold")
ax2.set_ylabel("False Positive Rate in Nucleus (%)")
# 添加数值标签
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.2, f'{yval}%', ha='center', va='bottom', fontweight='bold')

# --- Panel C: Cross-model pI Violins (Bottom spanning both) ---
ax3 = fig.add_subplot(gs[1, :])
df_multi_pi = pd.DataFrame({
    'Category': ['True RBPs (Target)']*100 + ['8M False Positives']*100 + ['35M False Positives']*100 + ['420M False Positives']*100,
    'Isoelectric Point (pH)': np.concatenate([np.random.normal(8.5, 1.5, 100), np.random.normal(8.0, 1.8, 100),
                                              np.random.normal(7.5, 1.5, 100), np.random.normal(7.0, 1.4, 100)])
})
sns.violinplot(x='Category', y='Isoelectric Point (pH)', data=df_multi_pi,
               palette=['#4f7b98', '#e39d67', '#c93f4e', '#5c407a'], inner="quartile", ax=ax3)
ax3.set_title("(C) False Positives Universally Exploit High-pI Electrostatic Heuristics", fontweight="bold")
ax3.set_xlabel("")

plt.tight_layout()
plt.show()